# 🧠 Meningioma Modelling Notebook

Consumes `output/datasets/` from the cleaning notebook.

DDA + EDA on **unimputed** data. Multivariable modelling on **imputed** data.


## 00. Setup

In [1]:
import pandas as pd
pd.set_option("display.max_columns", None)

from pathlib import Path

from IPython.display import display

from schema_infer import (
    infer_schema, print_schema_template, print_column_uniques, schema_summary, ColSpec,
)
from cleaning import format_table_for_display
from dda import run_dda
from missingness_resolution import load_unimputed_dataset
from eda import screen_associations
from diagnostic_accuracy import screen_diagnostic_accuracy
from inferential import run_inferential_stage, preview_multivariable_cases
from dataset_handoff import detect_imputation_method

OUTPUT_ROOT = Path("output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)  # do not wipe — reads cleaning outputs

from config import load

#🟧🟧🟧 None = all years; e.g. [2025] for one cohort year

ANALYSIS_YEARS: list[int] | None = None


## 01. Load prepared datasets

Requires `output/datasets/unimputed_df.parquet` and a modelling parquet from cleaning.


In [2]:
IMPUTATION_METHOD = detect_imputation_method(OUTPUT_ROOT)
if IMPUTATION_METHOD == "mice":
    print("Using MICE-imputed dataset.")
else:
    print("Using simple-imputed dataset.")

df = load_unimputed_dataset(OUTPUT_ROOT)
df.head()


Using MICE-imputed dataset.


,id,patient_code,entry_year,age,sex,histology_available,who_grade,progesterone_pos,ki67_pct,brain_invasion,hist_necrosis,mri_date,side,tumor_location,meningioma_count,max_diameter_cm,tumor_volume,base_modality,iv_contrast,tumor_episode,tumor_margin,dural_tail,capsular_enhancement,heterogeneous_enhancement,perifocal_edema,edema_volume_cm3,mass_effect,calcification,cystic_component,mri_necrosis,hemorrhage,hyperostosis,cortical_destruction,dwi_hyperintensity,t2_hyperintensity,t1_hypointensity,sinus_invasion,transfalcine_extension,adc_value,age_bins,high_grade,multiple_meningiomas,ki67_mid,ki67_group
0,2,070458-11352,2025,67.0,female,True,1,True,1-3,False,False,2025-06-27,right,skull_base,2,4.9,36.5,mri,True,primary,regular,False,True,False,True,5.0,True,True,False,False,False,False,False,True,True,True,no_invasion,False,0.88,60-69,False,True,2.0,low_le_4
1,3,230949-11093,2025,76.0,female,True,1,True,1-5,False,False,2025-09-05,midline,skull_base,1,2.8,6.86,mri_ct,True,primary,irregular,False,True,False,True,26.0,True,False,False,False,False,False,False,True,True,True,no_invasion,True,0.94,70-79,False,False,3.0,low_le_4
2,4,140352-11498,2025,73.0,female,True,2,True,25-30,True,True,2025-07-23,right,non_skull_base,1,4.7,40.9,mri_ct,True,recurrent,irregular,False,True,True,True,135.0,True,True,True,False,True,False,False,True,True,True,no_invasion,False,0.6,70-79,True,False,27.5,high_ge_10
3,5,151269-12200,2025,55.0,male,True,1,True,1-2,False,False,2025-08-04,right,non_skull_base,1,3.7,8.3,mri_ct,True,primary,irregular,True,True,False,True,24.0,True,True,False,False,False,False,False,True,True,True,no_invasion,False,1.2,50-59,False,False,1.5,low_le_4
4,6,270866-10213,2025,58.0,female,True,1,True,1-2,False,False,2025-12-27,right,non_skull_base,1,2.9,4.43,mri,True,primary,irregular,True,True,False,False,0.0,True,False,False,False,False,True,False,True,True,True,no_invasion,False,0.93,50-59,False,False,1.5,low_le_4


## 02. Reload schema

Re-run infer + overrides on the loaded cohort (same cells as monolithic notebook §04).


In [3]:
schema = infer_schema(df)
schema_summary(schema)


,column,kind,keep,datetime_bin,levels,nulls,note
0,id,id,True,None,None,None,
1,patient_code,id,True,None,None,None,
2,entry_year,ordinal,True,None,"[2018, 2019, 2020, 2021, 2022, 2023, 2024, 202...",None,
3,age,continuous,True,None,None,None,
4,sex,nominal,True,None,None,None,
5,histology_available,binary,True,None,None,None,
6,who_grade,ordinal,True,None,"[1, 2, 3]",None,
7,progesterone_pos,binary,True,None,None,None,
8,ki67_pct,text,True,None,None,None,
9,brain_invasion,binary,True,None,None,None,


In [4]:
print_schema_template(schema)


schema_overrides = {
    'id': ColSpec(name='id', kind='id'),
    'patient_code': ColSpec(name='patient_code', kind='id'),
    'entry_year': ColSpec(name='entry_year', kind='ordinal', ordered_levels=[2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]),
    'age': ColSpec(name='age', kind='continuous'),
    'sex': ColSpec(name='sex', kind='nominal'),
    'histology_available': ColSpec(name='histology_available', kind='binary'),
    'who_grade': ColSpec(name='who_grade', kind='ordinal', ordered_levels=['1', '2', '3']),
    'progesterone_pos': ColSpec(name='progesterone_pos', kind='binary'),
    'ki67_pct': ColSpec(name='ki67_pct', kind='text'),
    'brain_invasion': ColSpec(name='brain_invasion', kind='binary'),
    'hist_necrosis': ColSpec(name='hist_necrosis', kind='binary'),
    'mri_date': ColSpec(name='mri_date', kind='datetime'),
    'side': ColSpec(name='side', kind='nominal'),
    'tumor_location': ColSpec(name='tumor_location', kind='nominal'),
    'meningioma_count': ColSpec

In [5]:
#🟧🟧🟧 inspect raw values — use for nulls=() and replace={} below
print_column_uniques(df, schema)


📋 Column uniques — for nulls=() and replace={} in schema_overrides below

▸ id · id
  · 352 unique
  · '2' → 1
  · '3' → 1
  · '4' → 1
  · '5' → 1
  · '6' → 1
  · '8' → 1
  · '9' → 1
  · '10' → 1
  · … 344 more values

▸ patient_code · id
  · 352 unique
  · '070458-11352' → 1
  · '230949-11093' → 1
  · '140352-11498' → 1
  · '151269-12200' → 1
  · '270866-10213' → 1
  · '260962-11211' → 1
  · '201065-10534' → 1
  · '260471-11644' → 1
  · … 344 more values

▸ entry_year · ordinal
  · np.int64(2018) → 54
  · np.int64(2019) → 52
  · np.int64(2020) → 25
  · np.int64(2021) → 25
  · np.int64(2022) → 17
  · np.int64(2023) → 35
  · np.int64(2024) → 67
  · np.int64(2025) → 76
  · np.int64(2026) → 1

▸ age · continuous
  · 60 unique · 20.0 … 92.0

▸ sex · nominal
  · 'female' → 243
  · 'male' → 109

▸ histology_available · binary
  · np.False_ → 1
  · np.True_ → 351

▸ who_grade · ordinal
  · '1' → 247
  · '2' → 94
  · '3' → 11

▸ progesterone_pos · binary
  · np.False_ → 9
  · np.True_ → 342
  

In [6]:
#🟧🟧🟧 Edit overrides, then run

schema_overrides = {
    'id': ColSpec(name='id', kind='id'),
    'patient_code': ColSpec(name='patient_code', kind="id", keep=False),
    
    'entry_year': ColSpec(name='entry_year', kind='datetime', keep=False, datetime_bin='year'),
    'age': ColSpec(name='age', kind='continuous'),
    'sex': ColSpec(name='sex', kind='nominal', replace={0:"male", 1:"female",}),
    
    'who_grade': ColSpec(name='who_grade', kind='ordinal', ordered_levels=["1","2","3"]),
    
    'histology_available': ColSpec(name='histology_available', kind='binary', nulls=(2,), keep=False),
    'progesterone_pos': ColSpec(name='progesterone_pos', kind='binary', nulls=(2,)),
    'ki67_pct': ColSpec(name='ki67_pct', kind='text'),
    'brain_invasion': ColSpec(name='brain_invasion', kind='binary'),
    'hist_necrosis': ColSpec(name='hist_necrosis', kind='binary'),
    
    'mri_date': ColSpec(name='mri_date', kind='datetime', keep=False, datetime_bin='full'),
    
    'side': ColSpec(name='side', kind='nominal', replace={'1': "right", '2': "left", '3': "midline"}),
    'tumor_location': ColSpec(name='tumor_location', kind='nominal', replace={0: "non_skull_base", 1: "skull_base"}, nulls=(2,)),
    'meningioma_count': ColSpec(name='meningioma_count', kind='count'),
    'max_diameter_cm': ColSpec(name='max_diameter_cm', kind='continuous'),
    'tumor_volume': ColSpec(name='tumor_volume', kind='continuous'),
    
    'base_modality': ColSpec(name='base_modality', kind='nominal', replace={0: "mri", 1: "ct", 3: "mri_ct"}, keep=False),
    'iv_contrast': ColSpec(name='iv_contrast', kind='binary', keep=False),
    
    'tumor_episode': ColSpec(name='tumor_episode', kind='ordinal', replace={'0': "primary", '1': "recurrent"}, ordered_levels=["primary", "recurrent"], nulls=("multiplas",)),
    
    'tumor_margin': ColSpec(name='tumor_margin', kind='nominal', replace={1: "regular", 2: "irregular"}, nulls=(0,)),
    'dural_tail': ColSpec(name='dural_tail', kind='binary'),
    
    'capsular_enhancement': ColSpec(name='capsular_enhancement', kind='binary'),
    'heterogeneous_enhancement': ColSpec(name='heterogeneous_enhancement', kind='binary'),
    'dwi_hyperintensity': ColSpec(name='dwi_hyperintensity', kind='binary', nulls=('-',)),
    't2_hyperintensity': ColSpec(name='t2_hyperintensity', kind='binary', nulls=('-',)),
    't1_hypointensity': ColSpec(name='t1_hypointensity', kind='binary', nulls=('-',)),
    
    'perifocal_edema': ColSpec(name='perifocal_edema', kind='binary'),
    'edema_volume_cm3': ColSpec(name='edema_volume_cm3', kind='continuous'),
    
    'mass_effect': ColSpec(name='mass_effect', kind='binary'),
    'calcification': ColSpec(name='calcification', kind='binary'),
    'cystic_component': ColSpec(name='cystic_component', kind='binary'),
    'mri_necrosis': ColSpec(name='mri_necrosis', kind='binary'),
    'hemorrhage': ColSpec(name='hemorrhage', kind='binary', nulls=(2.0,)),
    'hyperostosis': ColSpec(name='hyperostosis', kind='binary'),
    'sinus_invasion': ColSpec(name='sinus_invasion', kind='ordinal', replace={0: "no_invasion", 1: "sinus_invasion", 2: "transsinus_extension"}, ordered_levels=["no_invasion", "sinus_invasion", "transsinus_extension"]),
    'cortical_destruction': ColSpec(name='cortical_destruction', kind='binary'),
    'transfalcine_extension': ColSpec(name='transfalcine_extension', kind='binary'),
    
    'adc_value': ColSpec(name='adc_value', kind='continuous'),
    }


In [7]:
load("03_schema_overrides").apply_schema_overrides(schema, schema_overrides, OUTPUT_ROOT)

## 03. Analysis configuration

Three separate lists — EDA screening uses the wide predictor pool; multivariable models use their own per-variant predictor sets.


In [8]:
# 🟧🟧🟧 Copy-pasteable column names from the loaded cohort
load("07_analysis").print_copy_pasteable_columns(df)

# Copy-paste into EDA_PREDICTORS / model variant lists (44 columns)
COLUMNS = [
    'id',
    'patient_code',
    'entry_year',
    'age',
    'sex',
    'histology_available',
    'who_grade',
    'progesterone_pos',
    'ki67_pct',
    'brain_invasion',
    'hist_necrosis',
    'mri_date',
    'side',
    'tumor_location',
    'meningioma_count',
    'max_diameter_cm',
    'tumor_volume',
    'base_modality',
    'iv_contrast',
    'tumor_episode',
    'tumor_margin',
    'dural_tail',
    'capsular_enhancement',
    'heterogeneous_enhancement',
    'perifocal_edema',
    'edema_volume_cm3',
    'mass_effect',
    'calcification',
    'cystic_component',
    'mri_necrosis',
    'hemorrhage',
    'hyperostosis',
    'cortical_destruction',
    'dwi_hyperintensity',
    't2_hyperintensity',
    't1_hypointensity',
    'sinus_invasion',
    'transfalcine_extension',
    'adc_value',
    'age_bins',
    'high_grade',
    'multiple_meningiomas',
    'ki67_mid',
    'ki67_group',
]


### 🎯 EDA


In [9]:
EDA_TARGETS = ['high_grade', 'progesterone_pos', 'brain_invasion', 'ki67_group', 'hist_necrosis']
EDA_PREDICTORS = [
    'age',
    'age_bins',
    'sex',

    #'who_grade', ==> TARGET
    #'high_grade', ==> TARGET

    #'progesterone_pos',
    #'ki67_pct',
    #'ki67_mid',
    #'ki67_group'
    #'brain_invasion',
    #'hist_necrosis',

    'side',
    'tumor_location',
    'meningioma_count',
    'multiple_meningiomas',
    'max_diameter_cm',
    'tumor_volume',

    'tumor_episode',
    'tumor_margin',
    'dural_tail',

    'perifocal_edema',
    'edema_volume_cm3',

    'mass_effect',
    'calcification',
    'cystic_component',
    'necrosis',
    'hemorrhage',
    'hyperostosis',
    'cortical_destruction',
    'sinus_invasion',
    'transfalcine_extension',

    'capsular_enhancement',
    'heterogeneous_enhancement',
    'dwi_hyperintensity',
    't2_hyperintensity',
    't1_hypointensity',

    'adc_value',
    ]

### 📚 Literature-based multivariable models


In [10]:
# 📚 Literature-based multivariable models — published predictor sets.
# Each variant gets its own EPV bar, forest plot, VIF table, and interpretation.
# Format: (id, title, link, target, [predictors])
LITERATURE_MODEL_VARIANTS = [
    # Research work: Predicting the grade of meningiomas by clinical–radiological features: A comparison of precontrast and postcontrast MRI
    # Authors: Yuan Yao, Yifan Xu, Shihe Liu, Feng Xue, Bao Wang, Shanshan Qin, Xiubin Sun, Jingzhen He
    # Link: https://www.frontiersin.org/journals/oncology/articles/10.3389/fonc.2022.1053089/full
    (
        "yao_et_al_2022",
        "Yao et al. 2022 | precontrast / semantic MRI model",
        "https://www.frontiersin.org/journals/oncology/articles/10.3389/fonc.2022.1053089/full",
        "high_grade",
        [
            "sex",
            "tumor_margin",
            "cystic_component",
            "perifocal_edema",
            "dural_tail",
        ],
    ),

    # Research work: Preoperative Prediction of Intracranial Meningioma Grade Using Conventional CT and MRI
    # Authors: T. Amano et al.
    # Link: https://www.cureus.com/articles/80763-preoperative-prediction-of-intracranial-meningioma-grade-using-conventional-ct-and-mri
    (
        "amano_et_al_2021_expanded_proxy",
        "Amano et al. 2021 expanded proxy | conventional CT/MRI + tumor burden",
        "https://www.cureus.com/articles/80763-preoperative-prediction-of-intracranial-meningioma-grade-using-conventional-ct-and-mri",
        "high_grade",
        [
            "tumor_location",
            "tumor_margin",
            "heterogeneous_enhancement",
            "perifocal_edema",
            "tumor_volume",
            "cortical_destruction",
        ],
    ),

    # Research work: The Role of Pre-Operative MRI for Prediction of High-Grade Intracranial Meningioma: A Retrospective Study
    # Authors: Kan Radeesri, Vitit Lekhavat
    # Link: https://journal.waocp.org/article_90552.html
    (
        "radeesri_lekhavat_2020",
        "Radeesri & Lekhavat 2020 | edema / necrosis MRI model",
        "https://journal.waocp.org/article_90552.html",
        "high_grade",
        [
            "perifocal_edema",
            "edema_volume_cm3",
            "mri_necrosis",
            "hemorrhage",
            "hyperostosis",
            "mass_effect",
        ],
    ),

    # Research work: Role of ADC values and ratios of MRI scan in differentiating typical, atypical and anaplastic meningiomas
    # Authors: M. Azeemuddin et al.
    # Link: https://pubmed.ncbi.nlm.nih.gov/30317276/
    (
        "azeemuddin_et_al_2018",
        "Azeemuddin et al. 2018 | diffusion-augmented MRI model",
        "https://pubmed.ncbi.nlm.nih.gov/30317276/",
        "high_grade",
        [
            "adc_value",
            "dwi_hyperintensity",
            "tumor_location",
            "tumor_margin",
            "perifocal_edema",
            "heterogeneous_enhancement",
            "sex",
        ],
    ),

    # Research work: Diagnostic nomogram model for predicting preoperative pathological grade of meningioma
    # Authors: Shijun Peng, Zhihua Cheng, Zhilin Guo
    # Link: https://tcr.amegroups.org/article/view/55552/html
    (
        "peng_cheng_guo_2021",
        "Peng, Cheng & Guo 2021 | interface / invasion model",
        "https://tcr.amegroups.org/article/view/55552/html",
        "high_grade",
        [
            "tumor_location",
            "tumor_margin",
            "sinus_invasion",
            "cortical_destruction",
            "mass_effect",
            "edema_volume_cm3",
            "hyperostosis",
        ],
    ),
]


### 🧪 Experimental multivariable models


In [11]:
# 🧪 Experimental multivariable models — your own predictor sets (independent of EDA_PREDICTORS).
# Add as many as you need. Each row is one model: (id, title, link, target, [predictors]).
# Model ids must be "experimental" or start with "experimental_" so the report groups them correctly.
EXPERIMENTAL_MODEL_VARIANTS = [
    (
        "experimental_model_1",
        "model 1 | high grade",
        "",
        "high_grade",
        [
            'cystic_component',
            'cortical_destruction',
            'dural_tail',
            'tumor_volume',
            'edema_volume_cm3',
            'hyperostosis',
            'mass_effect',
            'adc_value',
            'tumor_margin',
        ],
    ),
    (
        "experimental_model_2",
        "model 2 | high grade",
        "",
        "high_grade",
        [
            'dwi_hyperintensity',
            'sex',
            'heterogeneous_enhancement',
            'hemorrhage',
            'sinus_invasion',
            'age_bins',
            'calcification',
            't2_hyperintensity',
            't1_hypointensity',
            'transfalcine_extension',

        ],
    ),

    # Example — uncomment and edit to fit another outcome:
    # (
    #     "experimental_ki67",
    #     "meningioma_atypier experimental | Ki-67 group",
    #     "",
    #     "ki67_group",
    #     [
    #         "age_bins",
    #         "sex",
    #         "max_diameter_cm",
    #         "perifocal_edema",
    #     ],
    # ),
]


In [12]:
_c07 = load("07_analysis")

(
    EDA_TARGETS,
    EDA_PREDICTORS,
    EDA_POSITIVE_CLASS,
) = _c07.resolve_eda(df, EDA_TARGETS, EDA_PREDICTORS)

INFERENTIAL_MODEL_VARIANTS = _c07.resolve_inferential_variants(
    df,
    LITERATURE_MODEL_VARIANTS + EXPERIMENTAL_MODEL_VARIANTS,
)
(
    INFERENTIAL_TARGETS,
    INFERENTIAL_POSITIVE_CLASS,
) = _c07.resolve_inferential_targets(df, INFERENTIAL_MODEL_VARIANTS)


## 04. DDA on unimputed data

In [13]:
dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT)

for name in ("overall", "continuous", "categorical", "binary", "datetime", "id_text"):
    print(f"\n--- {name} ---")
    tbl = dda_tables.get(name)
    if tbl is None or tbl.empty:
        print("(none)")
    else:
        display(format_table_for_display(tbl))



--- overall ---


,n_rows,n_cols,n_cols_analysed,missing_cells_pct
0,352,44,38,0.6



--- continuous ---


,column,kind,n,n_unique,missing_pct,min,p_5th,median,mean,trimmed_mean,p_95th,max,mode,std,cv,iqr,skewness,kurtosis
0,age,continuous,352,60,0.0,20.00,40.00,65.00,63.11,63.70,81.00,92.0,71.00,12.68,0.20,17.25,-0.43,-0.14
1,meningioma_count,count,352,6,0.0,1.00,1.00,1.00,1.17,1.02,2.00,6.0,1.00,0.58,0.50,0.00,4.51,24.42
2,max_diameter_cm,continuous,352,88,0.0,0.20,1.79,3.80,4.06,3.95,7.30,9.2,1.80,1.73,0.43,2.50,0.51,-0.42
3,tumor_volume,continuous,329,282,6.5,0.30,1.60,14.70,27.62,21.57,98.76,168.0,2.00,31.80,1.15,31.42,1.66,2.31
4,edema_volume_cm3,continuous,333,182,5.4,0.00,0.00,4.48,20.85,13.43,93.60,197.0,0.00,32.74,1.57,29.20,2.16,5.06
5,adc_value,continuous,309,77,12.2,0.41,0.64,0.82,0.85,0.83,1.19,1.7,0.79,0.17,0.20,0.17,1.19,2.99
6,ki67_mid,continuous,352,29,0.0,1.00,1.00,2.50,4.34,2.97,17.50,55.0,1.00,5.88,1.35,3.00,3.92,21.29



--- categorical ---


,column,kind,ordered,n,n_unique,missing_pct,first_mode,first_mode_pct,second_mode,second_mode_pct,rarest,rarest_pct,max_class_imbalance,median_category,balance,entropy_bin
0,sex,nominal,False,352,2,0,female,69.0,,,male,31.0,2.23,,0.89,0.89
1,who_grade,ordinal,True,352,3,0,1,70.2,2,26.7,3,3.1,22.45,1,0.65,1.02
2,side,nominal,False,352,3,0,right,45.2,left,44.9,midline,9.9,4.54,,0.86,1.37
3,tumor_location,nominal,False,352,2,0,non_skull_base,54.5,,,skull_base,45.5,1.20,,0.99,0.99
4,tumor_episode,ordinal,True,352,2,0,primary,88.1,,,recurrent,11.9,7.38,primary,0.53,0.53
5,tumor_margin,nominal,False,352,2,0,regular,55.4,,,irregular,44.6,1.24,,0.99,0.99
6,sinus_invasion,ordinal,True,352,3,0,no_invasion,74.4,sinus_invasion,18.2,transsinus_extension,7.4,10.08,no_invasion,0.66,1.04
7,age_bins,ordinal,True,352,5,0,60-69,28.7,70-79,26.7,80+,8.2,3.48,60-69,0.95,2.21
8,ki67_group,ordinal,True,352,3,0,low_le_4,74.2,intermediate_5_9,15.9,high_ge_10,9.9,7.46,low_le_4,0.68,1.07



--- binary ---


,column,kind,ordered,n,n_unique,missing_pct,mode,mode_pct,rarest,rarest_pct,max_class_imbalance,balance,entropy_bin
0,progesterone_pos,binary,False,351,2,0.3,True,97.4,False,2.6,38.00,0.17,0.17
1,brain_invasion,binary,False,352,2,0.0,False,98.3,True,1.7,57.67,0.12,0.12
2,hist_necrosis,binary,False,352,2,0.0,False,90.1,True,9.9,9.06,0.47,0.47
3,dural_tail,binary,False,352,2,0.0,True,81.2,False,18.8,4.33,0.70,0.70
4,capsular_enhancement,binary,False,352,2,0.0,True,88.1,False,11.9,7.38,0.53,0.53
5,heterogeneous_enhancement,binary,False,352,2,0.0,True,58.8,False,41.2,1.43,0.98,0.98
6,perifocal_edema,binary,False,352,2,0.0,True,65.3,False,34.7,1.89,0.93,0.93
7,mass_effect,binary,False,352,2,0.0,True,86.1,False,13.9,6.18,0.58,0.58
8,calcification,binary,False,352,2,0.0,False,53.1,True,46.9,1.13,1.00,1.00
9,cystic_component,binary,False,352,2,0.0,False,78.7,True,21.3,3.69,0.75,0.75



--- datetime ---
(none)

--- id_text ---


,column,kind,n,missing_pct,n_unique
0,id,id,352,0,352
1,ki67_pct,text,352,0,42


## 05. EDA on unimputed data


Targets can be **binary**, **continuous**, **ordinal**, or **nominal** (from schema). The test depends on both outcome and predictor types — e.g. ordinal outcome × nominal predictor → χ²; continuous outcome × nominal predictor → Kruskal–Wallis.

| target kind   | continuous / count predictor | ordinal predictor | nominal / binary predictor |
|---------------|------------------------------|-------------------|----------------------------|
| binary        | Mann–Whitney U               | Spearman ρ        | χ² / Fisher                |
| continuous    | Spearman ρ                   | Spearman ρ        | Kruskal–Wallis             |
| ordinal       | Spearman ρ                   | Spearman ρ        | χ²                         |
| nominal       | Kruskal–Wallis               | χ²                | χ²                         |

`POSITIVE_CLASS` applies only to **binary** targets. Multivariable logistic (§16) remains **binary outcomes only**.

Per-target p-values are corrected with **Benjamini–Hochberg FDR**.


In [14]:
assoc = screen_associations(
    df, schema,
    targets=EDA_TARGETS,
    predictors=EDA_PREDICTORS,
    positive_class=EDA_POSITIVE_CLASS,
    fdr_alpha=0.05,
    output_root=OUTPUT_ROOT,
    )

diag_acc = screen_diagnostic_accuracy(
    df, schema,
    targets=EDA_TARGETS,
    predictors=EDA_PREDICTORS,
    positive_class=EDA_POSITIVE_CLASS,
    fdr_alpha=0.05,
    output_root=OUTPUT_ROOT,
)

#assoc[assoc['fdr_significant']]

/var/folders/c6/b51bn8410z541s8nqff2nxcm0000gn/T/ipykernel_7454/2131924962.py:10: UserWarning: Diagnostic accuracy skipped target 'ki67_group': requires binary outcome (kind=ordinal).
  diag_acc = screen_diagnostic_accuracy(


In [15]:
#🟧🟧🟧 Full table
#assoc

## 06. Multivariable modelling on imputed data

In [16]:
display(preview_multivariable_cases(
    schema,
    targets=INFERENTIAL_TARGETS,
    variants=INFERENTIAL_MODEL_VARIANTS,
    positive_class=INFERENTIAL_POSITIVE_CLASS,
    output_root=OUTPUT_ROOT,
))

,target,model_id,model_title,model_link,n_rows_total,n_complete_cases,n_rows_dropped,n_outcome_events,n_design_columns,epv
0,high_grade,amano_et_al_2021_expanded_proxy,Amano et al. 2021 expanded proxy | conventiona...,https://www.cureus.com/articles/80763-preopera...,352,352,0,105,6,17.5
1,high_grade,azeemuddin_et_al_2018,Azeemuddin et al. 2018 | diffusion-augmented M...,https://pubmed.ncbi.nlm.nih.gov/30317276/,352,352,0,105,7,15.0
2,high_grade,peng_cheng_guo_2021,"Peng, Cheng & Guo 2021 | interface / invasion ...",https://tcr.amegroups.org/article/view/55552/html,352,352,0,105,7,15.0
3,high_grade,radeesri_lekhavat_2020,Radeesri & Lekhavat 2020 | edema / necrosis MR...,https://journal.waocp.org/article_90552.html,352,352,0,105,6,17.5
4,high_grade,yao_et_al_2022,Yao et al. 2022 | precontrast / semantic MRI m...,https://www.frontiersin.org/journals/oncology/...,352,352,0,105,5,21.0
5,high_grade,experimental_model_1,model 1 | high grade,,352,352,0,105,9,11.7
6,high_grade,experimental_model_2,model 2 | high grade,,352,352,0,105,10,10.5


In [17]:
full_inferential_table = run_inferential_stage(
    schema,
    targets=INFERENTIAL_TARGETS,
    variants=INFERENTIAL_MODEL_VARIANTS,
    positive_class=INFERENTIAL_POSITIVE_CLASS,
    output_root=OUTPUT_ROOT,
    )
#full_inferential_table

## 07. Validation and model outputs

Inferential artifacts are written to `output/inferential/` by §06.


## 08. Build report.html


Builds `report.html` from artifacts already in `output/` (DDA, EDA, inferential).
Edit the settings cell, then run both cells.

- **`REPORT_TITLE` / `REPORT_AUTHOR`** — shown on the cover.
- **`REPORT_PATH`** — where to write the HTML file.
- **`analysis_years`** — optional cohort label suffix on the title (from §03).


In [18]:
REPORT_TITLE = "Non-invasive radiological biomarkers of meningiomas as a prognostic tool for predicting tumor histological grade"
REPORT_AUTHOR = "Arturs Balodis, Sigita Zālīte, Roberts Tumeļkāns, Valērija Aksjonova, Elizabete Stankeviča, Andris Zaguzovs"
REPORT_PATH = OUTPUT_ROOT / "report" / "report.html"

In [19]:
_c08 = load("08_report_settings")
_c08.run_report(
    output_root=OUTPUT_ROOT,
    report_title=REPORT_TITLE,
    report_author=REPORT_AUTHOR,
    report_path=REPORT_PATH,
    analysis_years=ANALYSIS_YEARS,
    eda_targets=EDA_TARGETS,
)
_c08.print_output_summary(OUTPUT_ROOT)

Report written: /Users/andriszaguzovs/TheLibraryOfCode/meningioma-atypier/heavy_machinery/output/report/report.html

📦 Pipeline outputs — /Users/andriszaguzovs/TheLibraryOfCode/meningioma-atypier/heavy_machinery/output
════════════════════════════════════════════════════════════════════════
📁 254 files · 16.5 MB total

🧹 Cleaning                  4 files ·  80.3 KB  (4 csv)
📋 Schema                    1 files ·   1.7 KB  (1 csv)
💾 Model datasets            3 files ·  88.2 KB  (1 json, 2 parquet)
📊 DDA                      48 files · 955.4 KB  (5 csv, 43 svg) — figures: 43, tables: 5
🕳️ Missingness              24 files · 759.3 KB  (13 csv, 4 json, 3 parquet, 2 png, 2 svg) — figures: 2, mice: 20, tables: 2
🔬 EDA                     142 files ·   5.3 MB  (2 csv, 140 svg) — figures: 140, tables: 2
🧮 Multivariable            30 files · 348.5 KB  (16 csv, 7 json, 7 svg) — figures: 7, tables: 23
🧾 Report                    1 files ·   9.0 MB  (1 html)
📁  Root                     1 files ·   